<a href="https://colab.research.google.com/github/tlamadon/abc-of-akm/blob/main/jep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hello, this is a notebook associated with the JEP article on the ABC of AKM. We provide simple simulation examples to make concrete the different points developed in the paper.

The first part is a quick dive in, while later section go behind the scene, explaining how the data is generated and diggs deeper in the analysis.

The notebook is fully self contained and you can look directly at all the code. Some of it is a bit lengthy so the code cell is closed to start with.

We hope this is useful and please provide any feedback directly to us.

TODO:
1. constructive example about between firm difference in wages versus wage gains in a move (using movers)
 - 1) simulate a network
 - 2) construct the outcome equation explicitely
 - 3) define the object of interest (as in the paper)
 - 4) show that difference in mean doesn't recover
 - 5) show that AKM does recover such difference
2. the bias example, also do 2 parametrizations.

In [ ]:
# @title Simulation code (hidden for quick dive)
# we generate a simple homophily network
# that we can later use to simulate data

# !apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm
from scipy.sparse import coo_matrix

import networkx as nx
import pandas as pd

import graphviz
from graphviz import Digraph
from IPython.display import Image
from IPython.display import display, Markdown

from string import Template
from typing import Any, Dict, List, Tuple, Optional

# plt.rcParams['text.usetex'] = True # TeX rendering
# plt.rcParams['mathtext.fontset'] = 'cm'
# plt.rcParams['font.family'] = 'STIXGeneral'

class Param:
    """Class to hold simulation parameters."""
    def __init__(self, **kwds: Any) -> None:
        self.__dict__.update({
            'rho': 1.0,
            'lambda1': 0.1,
            'sigma': 0.2,
            'ng': 10,
            'nj': 20
        })
        self.__dict__.update(kwds)

    def __str__(self) -> str:
        return ' '.join(f'{key}={value}' for key, value in self.__dict__.items())

    def __repr__(self) -> str:
        return str(self)

def matrix_stationary_distribution(transition_matrix: np.ndarray) -> np.ndarray:
    """Computes the stationary distribution of a Markov transition matrix."""
    eigvals, eigvecs = np.linalg.eig(transition_matrix.T)
    stationary_vec = eigvecs[:, np.isclose(eigvals, 1)]
    stationary_dist = stationary_vec[:, 0].real
    stationary_dist = np.maximum(0, np.sign(stationary_dist.sum()) * stationary_dist)
    stationary_dist /= stationary_dist.sum()
    return stationary_dist

def make_connected(dataset: pd.DataFrame) -> pd.DataFrame:
    """Filters the dataset to retain only the largest connected component of firms."""
    I = dataset['i'].to_numpy()
    J = dataset['j'].to_numpy()
    G = nx.Graph()
    for ii in range(1, len(I)):
        if (I[ii] != I[ii-1]):
            continue
        if (J[ii] != J[ii-1]):
            G.add_edge(J[ii-1], J[ii])
    largest_component = max(nx.connected_components(G), key=len)
    largest_component_list = list(largest_component)
    print("Largest connected component:", len(largest_component_list))
    dfc = dataset[dataset['j'].isin(largest_component_list)].copy()
    dfc.loc[:, 'j'] = pd.factorize(dfc['j'])[0]
    dfc.loc[:, 'i'] = pd.factorize(dfc['i'])[0]
    return dfc

def data_to_matrix(dataset: pd.DataFrame, normalize: bool = False) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Converts the dataset to design matrices for AKM estimation."""
    dataset = dataset.sort_values(by=['i', 't'])
    I = dataset['i'].to_numpy()
    J = dataset['j'].to_numpy()
    nn = len(I)
    nj = len(np.unique(J))
    ni = len(np.unique(I))
    Af = np.zeros((nn, nj-1))
    Aw = np.zeros((nn, ni))
    for n in range(nn):
        if J[n] < nj-1:
            Af[n, J[n]] = 1
        Aw[n, I[n]] = 1
    M = np.hstack((Af, Aw))
    return M, Af, Aw

class Model:
    """Model for simulating worker-firm matching and wages."""
    def __init__(self, p: Optional[Param] = None, **kwds: Any) -> None:
        if p is None:
            p = Param(**kwds)
        else:
            p.__dict__.update(kwds)

        self.lambda1: float = p.lambda1
        self.sigma: float = p.sigma
        self.rho: float = p.rho
        self.psi_j: np.ndarray = norm.ppf(np.linspace(1/p.nj, 1-1/p.nj, p.nj), loc=0, scale=1.0)
        self.alpha_g: np.ndarray = norm.ppf(np.linspace(1/p.ng, 1-1/p.ng, p.ng), loc=0, scale=1.0)
        self.ng: int = p.ng
        self.nj: int = p.nj
        self.H: Optional[np.ndarray] = None

    def tr_pr(self, alpha: float, psi1: float, psi2: float) -> float:
        """Calculates transition probability between psi1 and psi2 for a worker of type alpha."""
        return self.lambda1 * 1 / (1 + np.exp(1/self.rho * ((psi2-alpha)**2 - (psi1-alpha)**2)))

    def construct_transition_matrix(self, alpha: float, psi: np.ndarray) -> np.ndarray:
        """Constructs the full transition matrix for a given worker type."""
        n = len(psi)
        transition_matrix = np.zeros((n, n))
        for j in range(n):
            for jp in range(n):
                transition_matrix[j, jp] = self.tr_pr(alpha, psi[j], psi[jp])
        row_sums = transition_matrix.sum(axis=1, keepdims=True)
        transition_matrix /= row_sums
        return transition_matrix

    def stationary_distribution(self) -> np.ndarray:
        """Computes the stationary distributions for all worker types."""
        H = np.zeros((self.ng, self.nj))
        for g in range(self.ng):
            transition_matrix = self.construct_transition_matrix(self.alpha_g[g], self.psi_j)
            H[g, :] = matrix_stationary_distribution(transition_matrix)
        self.H = H
        return H

    def equilibrium_wages(self) -> np.ndarray:
        """Calculates equilibrium wages for all worker-firm combinations."""
        W = np.zeros((self.ng, self.nj))
        for g in range(self.ng):
            for j in range(self.nj):
                W[g, j] = self.alpha_g[g] + self.psi_j[j]
        return W

    def draw_residuals(self, dataset: pd.DataFrame) -> pd.DataFrame:
        """Draws wage residuals for a dataset."""
        dataset['epsilon'] = np.random.normal(0, 1, len(dataset)) * self.sigma
        dataset['w'] = dataset['alpha'] + dataset['psi'] + dataset['epsilon']
        return dataset

    def simulate(self, ni: int, nt: int, connected: bool = False) -> pd.DataFrame:
        """Simulates a panel dataset of worker-firm matches and wages."""
        if self.H is None:
            self.stationary_distribution()
        data: List[Dict[str, Any]] = []
        for i in range(ni):
            g = np.random.choice(range(self.ng), size=1)[0]
            j = np.random.choice(range(self.nj), size=1, p=self.H[g, :])[0]
            alpha_i = self.alpha_g[g] + np.random.normal(0, 1)
            data.append({'i': i, 'j': j, 't': 0, 'alpha': alpha_i})
            for t in range(1, nt):
                j2 = np.random.choice(range(self.nj), size=1)[0]
                pr_move = self.tr_pr(self.alpha_g[g], self.psi_j[j], self.psi_j[j2])
                if (j != j2) and (np.random.uniform(low=0, high=1) < pr_move):
                    j = j2
                data.append({'i': i, 'j': j, 't': t, 'alpha': alpha_i})
        dataset = pd.DataFrame(data)
        dataset['psi'] = self.psi_j[dataset['j']]
        dataset = self.draw_residuals(dataset)
        if connected:
            dataset = make_connected(dataset)
        return dataset

    def plot_wages_and_distribution(self) -> Tuple[plt.Figure, Any]:
        """Plots the stationary distribution and equilibrium wages."""
        H = self.stationary_distribution()
        W = self.equilibrium_wages()
        palette = sns.mpl_palette("tab10", 10)
        plt.rcParams['axes.prop_cycle'] = plt.cycler(color=palette)
        fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
        bottom = np.zeros(H.shape[1])
        H2 = H / H.sum(axis=0)
        for i in range(H.shape[0]):
            axes[0].fill_between(self.psi_j, bottom, bottom + H2[i, :])
            bottom += H2[i, :]
        axes[1].plot(self.psi_j, W.transpose())
        return fig, axes

def compute_akm(dataset: pd.DataFrame) -> pd.DataFrame:
    """Computes AKM estimates of worker and firm fixed effects."""
    M, Af, Aw = data_to_matrix(dataset)
    MM = np.matmul(M.transpose(), M)
    x = np.linalg.solve(MM, M.transpose() @ dataset['w'].to_numpy())
    psi_hat = Af @ x[range(Af.shape[1])]
    alpha_hat = Aw @ x[range(Af.shape[1], Aw.shape[1]+Af.shape[1])]
    dataset['alpha_hat'] = alpha_hat
    dataset['psi_hat'] = psi_hat
    return dataset

def variance_decomposition(data: pd.DataFrame, alpha_col: str = "alpha", psi_col: str = "psi", y_col: str = "w", format: str = "dict") -> Dict[str, float]:
    """Decomposes the variance of wages into components."""
    stats = {}
    stats["var_alpha"] = data[alpha_col].var()
    stats["var_psi"] = data[psi_col].var()
    stats["cov_alpha_psi"] = data[alpha_col].cov(data[psi_col])
    eps = data[y_col] - data[alpha_col] - data[psi_col]
    stats["var_eps"] = eps.var()
    return stats

def format_variance_decomposition(stats: Dict[str, float], text: str = "") -> str:
    """Formats variance decomposition stats into a Markdown table."""
    header = f"**{text}**\n\n" if text else ""
    table = (
        f"{header}"
        f"| $Var(\\alpha)$ | $Var(\\psi)$ | $2Cov(\\alpha,\\psi)$ | $Var(\\epsilon)$ |\n"
        f"|:---:|:---:|:---:|:---:|\n"
        f"| {stats['var_alpha']:.2f} | {stats['var_psi']:.2f} | {2*stats['cov_alpha_psi']:.2f} | {stats['var_eps']:.2f} |"
    )
    return table

def display_variance_decomposition(stats: Dict[str, float], text: str = "") -> None:
    """Displays the variance decomposition table as Markdown."""
    display(Markdown(format_variance_decomposition(stats, text)))

def abline(slope: float, intercept: float) -> None:
    """Plots a line from slope and intercept."""
    axes = plt.gca()
    x_vals = np.array(axes.get_xlim())
    y_vals = intercept + slope * x_vals
    plt.plot(x_vals, y_vals, '--')

## Quick dive, simulate and estimate AKM

We start with a quick dive, brushing over the details. We use the provided functions to create a parameter set and simulate a matched data set.

In [ ]:
np.random.seed(6344) # fixing the random seed

# initialize the model from the parameters
model = Model(lambda1=0.15, rho = 2.0 , sigma=1.0, ng=10, nj=100)

# simulate data from the model, impose connectedness
data = model.simulate(1_000, 5, connected=True)

data

The data contains is a panel of workers with the following columns:
- `i`: worker identifier
- `j`: firm identifier
- `t`: time period
- `alpha`: worker wage fixed heterogeneity
- `psi`: firm wage fixed heterogeneity
- `epsilon`: wage residual
- `w`: realized wage

### True variance decomposition

There isn't any estimates at this point. These are the true parameter values and residuals realizations. We can compute the "true" variance decomposition.



In [ ]:
# let's compute the variance decompostion and display it as a table
var_decomposition_true = variance_decomposition(data)
display_variance_decomposition(var_decomposition_true, text="True model decomposition")

### Compute AKM estimates

We can use the data, the columns `i`, `j` and `w` to compute the AKM estimates of $\psi$ and $\alpha$ and construct a decomposition using these estimates. When calling `compute_akm`, the estimated fixed effects are appended to the data in `alpha_hat` and `psi_hat`.  We can plot the estimated $\psi$ versus the true $\psi$.

In [ ]:
data = compute_akm(data)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

# Main scatter plot
plt.scatter(
    data['psi'],
    data['psi_hat'],
    alpha=0.4,
    edgecolors='w',
    linewidth=0.5,
    label='Firms'
)

# 45-degree reference line
lims = [
    np.min([plt.xlim(), plt.ylim()]),
    np.max([plt.xlim(), plt.ylim()])
]
plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='45° Line')

plt.xlabel(r'True $\psi$', fontsize=12)
plt.ylabel(r'Estimated $\psi$', fontsize=12)
plt.title('AKM Estimates: True vs Estimated Firm Effects', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

### Use AKM estimate to construct variance decomposition



In [ ]:
var_decomposition_hat = variance_decomposition(data, alpha_col="alpha_hat", psi_col="psi_hat", y_col="w")
display_variance_decomposition(var_decomposition_hat, text="Estimated model decomposition")

As we can see, while we simulated from a model which statifies all the assumption of AKM, the realized estimated decomposition is quite off. The variance of the firm effect is much larger and the covariance is much smaller, actually slightly negative.

### Monte-Carlo

We can simulate running the estimator on randomly drawn datasets to learn whether this was only due to the seed we used and this particular draw of the residuals.

We redraw the residuals, recompute the AKM estimates, and compare the average estimate to the true estimate. Because of the properties of OLS we expect this to be very well behaved in expectation.



In [ ]:
reps = [
    compute_akm(model.draw_residuals(data))['psi_hat']
    for rep in range(50)]
psi_hat_mean = np.vstack(reps).mean(axis=0)
psi_hat_sd = np.vstack(reps).std(axis=0)

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

# Error bar plot
plt.errorbar(data['psi'], psi_hat_mean, yerr=1.96 * psi_hat_sd,
             fmt='o', color='steelblue', ecolor='lightsteelblue',
             capsize=3, elinewidth=1, markeredgewidth=1,
             label=r'Mean Estimated $\psi$ (95% CI)')

# 45-degree reference line
lims = [np.min(data['psi']), np.max(data['psi'])]
plt.plot(lims, lims, 'r--', alpha=0.8, label='45° Reference')

plt.xlabel(r'True $\psi$', fontsize=12)
plt.ylabel(r'Estimated $\psi$', fontsize=12)
plt.title('Monte-Carlo: Stability of AKM Firm Effects', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

We see in the previous graph that the $\psi$ lign up very well with the true one on average. However there is a lot of variation around the estimated values. Such variation contributes the estimated variance.

# Addressing selection using AKM when measuring the role of firms in wage inequality

We start with an example of the usefulness of the AKM framework. We simulate data with a known wage equation. In such data we are interested in understanding the role of firms in the earnings inequality between workers.

In the following data, the wage of a worker $\alpha$ employed in firm $\psi$ is given by

$$y_{it} = \alpha_i + \psi_{j_{it}} + \epsilon_{it}$$

In [ ]:
np.random.seed(6344) # fixing the random seed

# initialize the model from the parameters
model = Model( lambda1=0.15, rho = 0.5 , sigma=1.0, ng=10, nj=100)

# simulate data from the model, impose connectedness
data = model.simulate(1_000, 5, connected=True)

data

We can use our simulated data to compute the variance of earnings across individuals, as well as the variance of earnings across firms.

In [ ]:
# Compute the mean by group and assign it to a new column
data['firm_wage'] = data.groupby('j')['w'].transform('mean')

total_var = data['w'].var()
between_var = data['firm_wage'].var()
share_percent = (between_var / total_var) * 100

print(f"Total variance:              {total_var:.3f}")
print(f"Between firm variance:       {between_var:.3f}")
print(f"Share between firm variance: {share_percent:.1f}%")

We find that 37% of the total wage variance can be attributed to differences in mean wage across firms. Meaning, when looking a the cross-section of firms, firms appear to pay very different mean wages.

Of course, the main question that AKM allows to address is whether these firm mean wage differences are large because:
1. different firms employ different workers: some firms employ very skilled workers and pay them for their skills, other firms hire mainly low skill workers. All firms pay the same wage to workers of the same skill.
2. different firms all employ similar workers, but some firms pay higher wages.

To answer such question, we run the AKM regression to recover both the $\alpha$ and the $\psi$ for each worker and firm (in the connected set).

In [ ]:
var_decomposition_true = variance_decomposition(data)
display_variance_decomposition(var_decomposition_true, text="True model decomposition")

Using our AKM decomposition we can then decompose the between variance to understand the role of firms.

$$Var ( E[w_{it} | j] ) = Var( \psi_{j(i,t)} ) +  Var( E[\alpha_{it} | j] ) + 2 Cov(\psi_{j(i,t)},  E[\alpha_{it} | j]) $$

In [ ]:
data['firm_alpha'] = data.groupby('j')['alpha'].transform('mean')
total_var = data['w'].var()

b_wage_var = data['firm_wage'].var()
b_comp_var = data['firm_alpha'].var()
b_pay_var = data['psi'].var()
b_cov_val = np.cov(data['psi'], data['firm_alpha'])[0, 1]

print(f"Between firm variance:       {b_wage_var:.3f} ({100*b_wage_var/total_var:4.1f}%)")
print(f"Between firm composition:    {b_comp_var:.3f} ({100*b_comp_var/total_var:4.1f}%)")
print(f"Between firm pay difference: {b_pay_var:.3f} ({100*b_pay_var/total_var:4.1f}%)")
print(f"Covariance between the two:  {b_cov_val:.3f} ({100*b_cov_val/total_var:4.1f}%)")

We note that:
 - the variance decomposition does add up to the total share of between firm variance
 - the decomposition highlights that the pay difference is actually less than half of what the between firm variance would suggest.

Perhaps more importantly, we could make this example much more sever by increasing the sorting of workers to firms.

As an exercise, change the value of $\rho$ in the parameters of the model, and rerun everything. Notice that you can bring the contribution of the firm pay difference to 0.

# How did we simulate the data?

We start with a couple of simple example. We generate homophily networks with 2 firms, 2 periods. Our first nework will be strongly segreated.



In [ ]:
# @title Creating move graph from dataset

np.random.seed(7236)
p2 = Param( lambda1=0.5, rho = 2.0 , sigma=0.2, ng=10, nj=5)
model = Model(p2)
dataset = model.simulate(10,5)

dataset = dataset.sort_values(by=['i', 't'])
I = dataset['i'].to_numpy()
J = dataset['j'].to_numpy()

g = Digraph("G", engine="neato", filename="ex.gv", format="png")
g.attr(size="7")

for ii in range(1,len(I)):
  if (I[ii] != I[ii-1]):
    continue
  if (J[ii] != J[ii-1]):
    g.edge(str(J[ii-1]), str(J[ii]), label=str(I[ii]))
g.render("mygraph")
fname = "mygraph.png"
Image(fname)

We generate a simple homophily network. EAch period, with probably $\lambda$ individuals get an opportunity to change employer. They meet a firm $j'$ at random and choose according to a simple logit rule whether to move.

A worker $\gamma_i$ moves from firm $j$ to firm $j'$ with probability

$$ \frac{ \lambda }{n_j} G\left( \frac{ v_{ij} - v_{ij'} }{\sigma} \right)$$

with value being

$$ v_{ij} = (\gamma_i - \eta_{j})^2 $$

when working at firm $j$ we assume a log linear wage equation:

$$ w_{it} = \alpha_i + \psi_{j(i,t)} + \epsilon_{it} $$

where we draw the $\epsilon_{it}$ completely indepently at each $(i,t)$. We draw the $\alpha_i$ in relation to $\gamma_i$, when unerlated, we have no sorting, when positively related, we get positive sorting, and when negative, we get negative sorting. We use
$$\begin{align}
\alpha_i & = \rho \gamma_i + (1-\rho) u_i \\
\psi_j  & = \rho \eta_i + (1-\rho) z_i
\end{align}$$

We can see in the raw data (below) that indeed invidual 0 moves from firm 1 to firm 3 between t=1 and t=2.

In [ ]:
dataset.head(10)

Next we show what the matrix looks like for the mobility of the current dataset. You can see that the first 2 rows have ones for firm 1 (index starts at 0 in pyton) followed by ones in the fourth column.

In [ ]:
M, Af, Aw = data_to_matrix(dataset)
Af[range(5),:]

# Two way fixed effect estimator

We construct the matrices associated with the mobility of the workers. We then compute the least square estimates. Finally we compare the predicted $\psi$ to the true $\psi$ values. We do observe some noise.

In [ ]:
np.random.seed(6344)
p3 = Param( lambda1=0.5, rho = 2.0 , sigma=0.2, ng=10, nj=100)
model = Model(p3)
dataset = model.simulate(1_000,5,connected=True)

M, Af, Aw = data_to_matrix(dataset)
MM = np.matmul(M.transpose(),M)
x = np.linalg.solve(MM, M.transpose() @ dataset['w'].to_numpy())

psi_jt = dataset['psi'].to_numpy()
alpha_jt = dataset['alpha'].to_numpy()
psi_hat = Af @ x[range(Af.shape[1])]
alpha_hat = Aw @ x[ range( Af.shape[1], Aw.shape[1]+Af.shape[1])]

In [ ]:
plt.scatter(
    psi_jt,
    psi_hat)
plt.show()

In [ ]:
def get_cov(lambda1):
  ptmp = Param(lambda1 = lambda1, rho = 2.0 ,sigma = 0.2, ng = 10, nj = 100)
  model = Model(ptmp)
  dataset = model.simulate(1_000,5,connected=True)

  M, Af, Aw = data_to_matrix(dataset)
  MM = np.matmul(M.transpose(),M)
  x = np.linalg.solve(MM, M.transpose() @ dataset['w'].to_numpy())

  psi_jt = dataset['psi'].to_numpy()
  alpha_jt = dataset['alpha'].to_numpy()
  psi_hat = Af @ x[range(Af.shape[1])]
  alpha_hat = Aw @ x[ range( Af.shape[1], Aw.shape[1]+Af.shape[1])]
  return(np.cov(alpha_hat, psi_hat)[0,1])

lambda1_vals = np.linspace(0.1,0.4,10)
covs = [get_cov(l) for l in lambda1_vals]

plt.plot(lambda1_vals, covs)
plt.xlabel('lambda1')

In [ ]:


template = Template(r"Hello, $name! Welcome to $place and $\alpha$")
formatted = template.safe_substitute(name="Alice", place="Wonderland")
print(formatted)


<div style="margin-top: 100px; margin-bottom: 20px;">This block has custom spacing.</div>
